# Within-App Adaptation Across Contexts

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

DATA_CANDIDATES = [
    Path("../data/mhealth_apps_metrics.csv"),
    Path("./mhealth_apps_metrics.csv"),
]
metrics_results = next((p for p in DATA_CANDIDATES if p.exists()), None)
if metrics_results is None:
    raise FileNotFoundError(
        "Could not find the metrics CSV. Checked: " + ", ".join(map(str, DATA_CANDIDATES))
    )
print(f"Using dataset: {metrics_results}")

REGION_MAP = {
    "us": "North America", "ca": "North America", "gl":"North America",
    "mx": "Latin America", "br": "Latin America", "ar": "Latin America", "cl": "Latin America",
    "de": "Europe", "fr": "Europe", "gb": "Europe", "pl": "Europe",
    "tr": "Middle East", "il": "Middle East", "sa": "Middle East", "ae": "Middle East",
    "in": "Asia", "jp": "Asia", "sg": "Asia", "kr": "Asia",
    "za": "Africa", "mz": "Africa", "ng": "Africa", "ke": "Africa",
    "au": "Oceania", "nz": "Oceania", "pg": "Oceania"
}

COUNTRY_LABEL_MAP = {
    "us": "United States", "ca": "Canada", "gl":"Greenland",
    "mx": "Mexico", "br": "Brazil", "ar": "Argentina", "cl": "Chile",
    "de": "Germany", "fr": "France", "gb": "Great Britain", "pl": "Poland",
    "tr": "Turkey", "il": "Israel", "sa": "Saudi Arabia", "ae": "United Arab Emirates",
    "in": "India", "jp": "Japan", "sg": "Singapore", "kr": "South Korea",
    "za": "South Africa", "mz": "Mozambique", "ng": "Nigeria", "ke": "Kenya",
    "au": "Australia", "nz": "New Zealand", "pg": "Papua New Guinea"
}

REGION_ORDER = [
    "North America", "Latin America", "Europe",
    "Middle East", "Asia", "Africa", "Oceania"
]

region_palette = plt.cm.tab10(np.linspace(0, 1, len(REGION_ORDER)))
REGION_COLORS = {region: region_palette[i] for i, region in enumerate(REGION_ORDER)}
category_palette = plt.cm.tab20(np.linspace(0, 1, 20))


In [ ]:
import ast
import re
from itertools import combinations
from functools import lru_cache

import numpy as np
import pandas as pd

# Canonical parsing helpers
INVALID_TOKENS = {"", "nan", "none", "null", "no", "n/a", "na", "no content", "not mentioned", "not applicable", "not at all"}

CANONICAL_PATTERNS = [
    ("email", [r"email", r"e mail"]),
    ("phone", [r"phone", r"telephone", r"mobile number"]),
    ("name", [r"full name", r"first name", r"last name", r"user name", r"username", r"\bname\b"]),
    ("location", [r"location", r"gps", r"address"]),
    ("device_id", [r"device id", r"advertising id", r"identifier", r"aaid", r"idfa"]),
    ("ip_address", [r"ip address", r"internet protocol"]),
    ("health", [r"health", r"medical", r"clinical", r"diagnos", r"symptom", r"medication"]),
    ("fitness", [r"fitness", r"exercise", r"activity", r"steps", r"heart rate"]),
    ("biometric", [r"biometric", r"fingerprint", r"face"]),
    ("financial", [r"financial", r"payment", r"credit card", r"bank", r"purchase"]),
    ("contact", [r"contact", r"address book"]),
    ("photo_video", [r"photo", r"video", r"image", r"media"]),
    ("audio", [r"audio", r"microphone", r"voice"]),
    ("calendar", [r"calendar"]),
    ("app_activity", [r"app activity", r"app interactions", r"usage data", r"in app activity"]),
    ("browsing_history", [r"browsing history", r"web history"]),
    ("crash_logs", [r"crash", r"diagnostic"]),
    ("performance", [r"performance", r"analytics"]),
]


def clean_text(x):
    t = str(x).strip().lower()
    t = re.sub(r"[_\-/]+", " ", t)
    t = re.sub(r"[^a-z0-9&+ ]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def canonicalize_token(x):
    t = clean_text(x)
    if t in INVALID_TOKENS:
        return None
    for canon, pats in CANONICAL_PATTERNS:
        if any(re.search(p, t) for p in pats):
            return canon
    return t if len(t.split()) <= 4 else None


def maybe_literal(x):
    if isinstance(x, (list, tuple, set, dict)):
        return x
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s or s.lower() in INVALID_TOKENS:
        return None
    if s[0] in "[{(" and s[-1] in "]})":
        try:
            return ast.literal_eval(s)
        except Exception:
            return s
    return s


@lru_cache(maxsize=200000)
def parse_set_cached(s):
    val = maybe_literal(s)
    if val is None:
        return frozenset()
    if isinstance(val, dict):
        raw = val.keys()
    elif isinstance(val, (list, tuple, set)):
        raw = val
    else:
        raw = re.split(r"[;,\n]", str(val))
    out = {canonicalize_token(v) for v in raw}
    return frozenset(v for v in out if v)


def parse_set(x):
    if isinstance(x, (list, tuple, set, dict)):
        return set(parse_set_cached(str(x)))
    if pd.isna(x):
        return set()
    return set(parse_set_cached(str(x)))


def parse_domain_set(x):
    val = maybe_literal(x)
    if val is None:
        return set()
    if isinstance(val, (list, tuple, set)):
        raw = val
    else:
        raw = re.split(r"[;,\n]", str(val))
    return {str(v).strip().lower() for v in raw if str(v).strip()}


def parse_freq_map(x):
    val = maybe_literal(x)
    if val is None:
        return {}
    if isinstance(val, dict):
        items = val.items()
    else:
        items = []
        for part in str(val).split(";"):
            if ":" not in part:
                continue
            k, v = part.rsplit(":", 1)
            items.append((k, v))
    out = {}
    for k, v in items:
        canon = canonicalize_token(k)
        if not canon:
            continue
        try:
            out[canon] = out.get(canon, 0.0) + float(v)
        except Exception:
            continue
    return out


def normalize_freq_map(mp):
    total = sum(max(float(v), 0.0) for v in mp.values())
    if total <= 0:
        return {}
    return {k: max(float(v), 0.0) / total for k, v in mp.items()}


def coerce_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def normalized_l1_distance(a, b):
    keys = set(a) | set(b)
    if not keys:
        return 0.0
    num = sum(abs(float(a.get(k, 0.0)) - float(b.get(k, 0.0))) for k in keys)
    den = sum(abs(float(a.get(k, 0.0))) + abs(float(b.get(k, 0.0))) for k in keys)
    return 0.0 if den == 0 else num / den


def merge_freq_maps(series):
    merged = {}
    for mp in series:
        if not isinstance(mp, dict):
            continue
        for k, v in mp.items():
            merged[k] = merged.get(k, 0.0) + float(v)
    return merged


def build_country_feature(country_df):
    feat = {}
    for state in ["pre", "post"]:
        merged = merge_freq_maps(country_df.get(f"{state}_type_frequencies_map", pd.Series(dtype=object)))
        for k, v in normalize_freq_map(merged).items():
            feat[f"{state}_freq::{k}"] = v
        set_col = f"{state}_observed_data_types_set"
        if set_col in country_df:
            types = set().union(*country_df[set_col].tolist()) if len(country_df) else set()
            for t in types:
                feat[f"{state}_type::{t}"] = 1.0
    return feat


def compute_app_as(app_df):
    features = {c: build_country_feature(g) for c, g in app_df.groupby("country", dropna=True)}
    countries = sorted(features)
    if len(countries) < 2:
        return np.nan
    dists = [normalized_l1_distance(features[a], features[b]) for a, b in combinations(countries, 2)]
    return float(np.mean(dists)) if dists else np.nan

# Load and prepare dataset
df = pd.read_csv(metrics_results, low_memory=False)
print("Raw shape:", df.shape)

if "country" not in df.columns or "app_id" not in df.columns:
    raise ValueError("Dataset must contain app_id and country columns.")

df["country"] = df["country"].astype(str).str.strip().str.lower()
df["country_label"] = df["country"].map(COUNTRY_LABEL_MAP).fillna(df.get("country_label"))
df["region"] = df["country"].map(REGION_MAP).fillna(df.get("region"))

if "categories" in df.columns:
    df["category"] = df["categories"].fillna("Unknown").astype(str).str.strip()
elif "category" in df.columns:
    df["category"] = df["category"].fillna("Unknown").astype(str).str.strip()
else:
    df["category"] = "Unknown"

for col in [
    "app_country_ADII", "app_country_PCLR", "ADII", "PCLR", "DGI", "AS",
    "app_country_pre_sensitive_instances", "app_country_total_sensitive_instances",
    "num_permissions", "num_dangerous_permissions", "num_trackers", "downloads_int", "average_score",
]:
    if col in df.columns:
        df[col] = coerce_numeric(df[col])

if "app_country_ADII" in df.columns:
    df["ADII"] = df["app_country_ADII"]
if "app_country_PCLR" in df.columns:
    df["PCLR"] = df["app_country_PCLR"]

for col in ["pre_type_frequencies", "post_type_frequencies"]:
    if col in df.columns:
        df[f"{col}_map"] = df[col].map(parse_freq_map)
    elif f"{col}_map" in df.columns:
        df[f"{col}_map"] = df[f"{col}_map"].map(parse_freq_map)

for col in ["pre_observed_data_types", "post_observed_data_types"]:
    if col in df.columns:
        df[f"{col}_set"] = df[col].map(parse_set)
    elif f"{col}_set" in df.columns:
        df[f"{col}_set"] = df[f"{col}_set"].map(parse_set)

# Canonicalized DGI: compare observed runtime data types against disclosed Data
OBSERVED_COLUMNS = ["pre_observed_data_types", "post_observed_data_types", "pre_PII", "post_PII", "pre_PHI", "post_PHI", "pre_OTHER", "post_OTHER"]
DISCLOSURE_COLUMNS = ["data_safety_data_shared", "data_safety_data_collected"]

for col in OBSERVED_COLUMNS + DISCLOSURE_COLUMNS:
    if col in df.columns:
        df[f"{col}_canon_set"] = df[col].map(parse_set)

obs_cols = [f"{c}_canon_set" for c in OBSERVED_COLUMNS if f"{c}_canon_set" in df.columns]
disc_cols = [f"{c}_canon_set" for c in DISCLOSURE_COLUMNS if f"{c}_canon_set" in df.columns]

observed_sets, disclosed_sets, missing_sets, misleading_sets = [], [], [], []
for row in df[obs_cols + disc_cols].itertuples(index=False, name=None):
    row_vals = dict(zip(obs_cols + disc_cols, row))
    observed = set().union(*(row_vals[c] for c in obs_cols if isinstance(row_vals[c], set))) if obs_cols else set()
    disclosed = set().union(*(row_vals[c] for c in disc_cols if isinstance(row_vals[c], set))) if disc_cols else set()
    missing = observed - disclosed
    misleading = disclosed - observed
    observed_sets.append(sorted(observed))
    disclosed_sets.append(sorted(disclosed))
    missing_sets.append(sorted(missing))
    misleading_sets.append(sorted(misleading))

dgi_df = pd.DataFrame({
    "observed_set_final": observed_sets,
    "disclosed_set_final": disclosed_sets,
    "missing_set": missing_sets,
    "misleading_set": misleading_sets,
}, index=df.index)
dgi_df["observed_count"] = dgi_df["observed_set_final"].str.len()
dgi_df["disclosed_count"] = dgi_df["disclosed_set_final"].str.len()
dgi_df["missing_count"] = dgi_df["missing_set"].str.len()
dgi_df["misleading_count"] = dgi_df["misleading_set"].str.len()
dgi_df["DGI"] = np.where(dgi_df["observed_count"].gt(0), dgi_df["missing_count"] / dgi_df["observed_count"], np.nan)

df = df.drop(columns=["DGI", "observed_count", "disclosed_count", "missing_count", "misleading_count", "observed_set_final", "disclosed_set_final"], errors="ignore")
df = pd.concat([df, dgi_df], axis=1)
print("Canonicalized DGI sanity check passed:", bool(((df.loc[df["observed_count"] > 0, "DGI"] - df.loc[df["observed_count"] > 0, "missing_count"] / df.loc[df["observed_count"] > 0, "observed_count"]).abs() < 1e-10).all()))


as_cols = [
    "app_id", "country",
    "pre_type_frequencies_map", "post_type_frequencies_map",
    "pre_observed_data_types_set", "post_observed_data_types_set",
]
as_cols = [c for c in as_cols if c in df.columns]
country_feature_records = []
for (app_id, country), g in df[as_cols].groupby(["app_id", "country"], dropna=True, sort=False):
    feat = build_country_feature(g)
    country_feature_records.append((app_id, country, feat))

features_by_app = {}
for app_id, country, feat in country_feature_records:
    features_by_app.setdefault(app_id, {})[country] = feat

as_rows = []
for app_id, feature_map in features_by_app.items():
    countries = sorted(feature_map)
    if len(countries) < 2:
        as_rows.append((app_id, np.nan))
        continue
    dists = [normalized_l1_distance(feature_map[a], feature_map[b]) for a, b in combinations(countries, 2)]
    as_rows.append((app_id, float(np.mean(dists)) if dists else np.nan))

app_as = pd.DataFrame(as_rows, columns=["app_id", "AS"])
df = df.drop(columns=["AS"], errors="ignore").merge(app_as, on="app_id", how="left")

required_metrics = ["ADII", "DGI", "PCLR"]
missing_required = [c for c in required_metrics if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required metric columns after preparation: {missing_required}")

analysis_df = df.dropna(subset=["ADII", "DGI", "PCLR"]).copy()
analysis_df = analysis_df[analysis_df["region"].notna() & analysis_df["country_label"].notna()].copy()

print("Analysis shape:", analysis_df.shape)
print("Unique apps:", analysis_df["app_id"].nunique())


**Note on AS/DGI in this notebook vs. the rest of the pipeline:** the cell above computes its own `AS` and `DGI` using a *canonicalized* token pipeline (`canonicalize_token`, bucketing free-text data-type mentions into ~18 broad categories and L1-normalizing frequency maps), which differs from the raw-token computation shared by `01_privacy_metrics.ipynb`, `06`, `08`, `09`, and `10`. This was measured to produce materially different values (e.g. mean AS roughly 0.11 here vs. roughly 0.38 under the shared definition) -- the two are **not directly comparable**. This notebook's adaptation-tier thresholds and groupings are internally consistent (computed from its own AS throughout), but any cross-reference to "AS" or "DGI" values quoted from another notebook should be read as referring to a different, non-interchangeable metric definition unless explicitly reconciled.

In [ ]:
print("Before shape:", df.shape)
print("After shape:", analysis_df.shape)

# -----------------------------
# Unique app counts per country
# -----------------------------
country_app_counts = (
    analysis_df.groupby(["region", "country_label"])["app_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="unique_app_count")
)

# country_app_counts

In [ ]:

summary = pd.DataFrame({
    "non_null": analysis_df[["ADII", "DGI", "PCLR", "AS"]].notna().sum(),
    "mean": analysis_df[["ADII", "DGI", "PCLR", "AS"]].mean(),
    "median": analysis_df[["ADII", "DGI", "PCLR", "AS"]].median(),
    "std": analysis_df[["ADII", "DGI", "PCLR", "AS"]].std(),
    "min": analysis_df[["ADII", "DGI", "PCLR", "AS"]].min(),
    "max": analysis_df[["ADII", "DGI", "PCLR", "AS"]].max(),
}).round(4)
summary


In [ ]:
country_level = analysis_df.copy()

# Rebuild app-level metrics from the prepared analysis dataframe.
metric_agg = {"ADII": "mean", "DGI": "mean", "PCLR": "mean", "AS": "first"}
meta_agg = {
    "country_label": "nunique",
    "category": lambda s: s.dropna().mode().iat[0] if not s.dropna().mode().empty else np.nan,
    "num_permissions": "mean",
    "num_dangerous_permissions": "mean",
    "num_trackers": "mean",
    "downloads_int": "mean",
    "average_score": "mean",
}
agg_dict = {k: v for k, v in {**metric_agg, **meta_agg}.items() if k in analysis_df.columns}
app_level = analysis_df.groupby("app_id", as_index=False).agg(agg_dict)
app_level = app_level.rename(columns={"country_label": "country_count"})
app_level_all = app_level.copy()

print("Country-level rows:", len(country_level))
print("App-level rows:", len(app_level_all))
print("App-level metric columns:", [c for c in ["ADII", "DGI", "PCLR", "AS"] if c in app_level_all.columns])
app_level = app_level_all.copy()



## Figures — Within-App Privacy Adaptation Analysis

This section analyzes cross-country within-app privacy adaptation using the app-level Adaptation Score (AS). It:
- groups apps into adaptation tiers,
- examines how adaptation relates to privacy risk and transparency,
- identifies which app categories and regions are most associated with adaptive behavior,
- highlights apps that are both adaptive and privacy-invasive.

In [ ]:
# -------------------------------------------------------------------
# Within-App Privacy Adaptation Analysis
# Primary grouping: k-means on normalized-frequency AS
# -------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

MIN_COUNTRIES = 3
RANDOM_STATE = 42
N_CLUSTERS = 3

adaptation_order = ["Stable", "Partially adaptive", "Highly adaptive"]
group_colors = {
    "Stable": "#4C78A8",
    "Partially adaptive": "#F58518",
    "Highly adaptive": "#E45756",
}
region_color_map = REGION_COLORS

def mode_or_first(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iat[0] if not m.empty else s.iloc[0]

adaptation_app_level = (
    analysis_df.groupby("app_id", as_index=False)
    .agg(
        category=("category", mode_or_first),
        downloads_int=("downloads_int", "median"),
        average_score=("average_score", "median"),
        num_trackers=("num_trackers", "median"),
        num_permissions=("num_permissions", "median"),
        num_dangerous_permissions=("num_dangerous_permissions", "median"),
        top_grossing=("top_grossing", mode_or_first),
        free=("free", mode_or_first),
        offersIAP=("offersIAP", mode_or_first),
        ADII_mean=("ADII", "mean"),
        ADII_median=("ADII", "median"),
        DGI_mean=("DGI", "mean"),
        DGI_median=("DGI", "median"),
        PCLR_mean=("PCLR", "mean"),
        PCLR_median=("PCLR", "median"),
        AS=("AS", "first"),
        n_countries=("country", "nunique"),
        country_count=("country_label", "nunique"),
    )
    .copy()
)

adaptation_app_level = adaptation_app_level.loc[
    (adaptation_app_level["n_countries"] >= MIN_COUNTRIES) & adaptation_app_level["AS"].notna()
].copy()

if adaptation_app_level.empty:
    raise ValueError(f"No apps remain after filtering for n_countries >= {MIN_COUNTRIES} and non-null AS.")
if adaptation_app_level["AS"].nunique() < N_CLUSTERS:
    raise ValueError(f"AS has only {adaptation_app_level['AS'].nunique()} unique values; cannot fit {N_CLUSTERS} clusters reliably.")

print(f"Apps retained after country filtering (n_countries >= {MIN_COUNTRIES}): {len(adaptation_app_level)}")
print(f"Median countries observed per app: {adaptation_app_level['n_countries'].median():.1f}")

# K-means is the only adaptation-group definition used downstream.
# A lightweight deterministic 1-D implementation avoids unnecessary runtime
# overhead from repeatedly importing/fitting general-purpose clustering.
def kmeans_1d(values, k=3, max_iter=100, random_state=42):
    values = np.asarray(values, dtype=float).ravel()
    if np.unique(values).size < k:
        raise ValueError(f"Need at least {k} unique AS values for k-means grouping.")
    # Quantile initialization is deterministic and stable for one-dimensional AS.
    qs = np.linspace(0, 1, k + 2)[1:-1]
    centers = np.quantile(values, qs)
    labels = np.zeros(values.shape[0], dtype=int)
    for _ in range(max_iter):
        new_labels = np.argmin(np.abs(values[:, None] - centers[None, :]), axis=1)
        new_centers = np.array([
            values[new_labels == j].mean() if np.any(new_labels == j) else centers[j]
            for j in range(k)
        ])
        if np.array_equal(new_labels, labels) and np.allclose(new_centers, centers):
            labels = new_labels
            centers = new_centers
            break
        labels = new_labels
        centers = new_centers
    return labels, centers

labels, centers = kmeans_1d(adaptation_app_level["AS"].to_numpy(), k=N_CLUSTERS, random_state=RANDOM_STATE)
adaptation_app_level["adaptation_cluster"] = labels

centroid_df = pd.DataFrame({
    "cluster": np.arange(N_CLUSTERS),
    "centroid_AS": centers,
}).sort_values("centroid_AS").reset_index(drop=True)

cluster_to_label = {
    centroid_df.loc[0, "cluster"]: "Stable",
    centroid_df.loc[1, "cluster"]: "Partially adaptive",
    centroid_df.loc[2, "cluster"]: "Highly adaptive",
}

adaptation_app_level["adaptation_group"] = adaptation_app_level["adaptation_cluster"].map(cluster_to_label)
adaptation_app_level["adaptation_group"] = pd.Categorical(
    adaptation_app_level["adaptation_group"], categories=adaptation_order, ordered=True
)

print("\nK-means centroids used for primary AS groups:")
display(centroid_df)

group_counts = (
    adaptation_app_level["adaptation_group"]
    .value_counts(dropna=False)
    .reindex(adaptation_order, fill_value=0)
    .rename_axis("adaptation_group")
    .reset_index(name="apps")
)
group_counts["percent"] = 100 * group_counts["apps"] / group_counts["apps"].sum()
print("\nPrimary adaptation groups (K-means):")
display(group_counts)

group_stats = (
    adaptation_app_level.groupby("adaptation_group", observed=False)
    .agg(
        apps=("app_id", "count"),
        AS_mean=("AS", "mean"),
        AS_median=("AS", "median"),
        AS_std=("AS", "std"),
        AS_min=("AS", "min"),
        AS_max=("AS", "max"),
        ADII_mean=("ADII_mean", "mean"),
        DGI_mean=("DGI_mean", "mean"),
        PCLR_mean=("PCLR_mean", "mean"),
        countries_mean=("n_countries", "mean"),
        trackers_mean=("num_trackers", "mean"),
        permissions_mean=("num_permissions", "mean"),
    )
    .reindex(adaptation_order)
    .reset_index()
)
print("\nGroup descriptive statistics:")
display(group_stats.round(4))

# -------------------------------------------------------------------
# Region-level adaptation signal
# -------------------------------------------------------------------
# IMPORTANT: We should not use raw app-country observation counts for panel (c).
# In this dataset, most apps are executed in the same country/region panel.
# Therefore, counting app-country rows mechanically gives almost identical regional
# shares for every adaptation group and does not reveal where adaptation occurs.
#
# Instead, compute a region-level adaptation contribution: for each app-country
# feature vector, measure its normalized L1 distance from that app's equal-weight
# cross-country centroid. This keeps the within-app design and asks which regions
# deviate more from each app's overall behavior.

app_group_lookup = adaptation_app_level[["app_id", "adaptation_group"]].copy()
retained_app_ids = set(app_group_lookup["app_id"])
country_meta = analysis_df[["app_id", "country", "country_label", "region"]].drop_duplicates()

def mean_feature_vector(feature_dicts):
    keys = sorted(set().union(*(d.keys() for d in feature_dicts))) if feature_dicts else []
    if not keys:
        return {}
    return {k: float(np.mean([d.get(k, 0.0) for d in feature_dicts])) for k in keys}

region_signal_records = []
for app_id, per_country_features in features_by_app.items():
    if app_id not in retained_app_ids:
        continue
    countries = sorted(per_country_features)
    if len(countries) < MIN_COUNTRIES:
        continue
    centroid = mean_feature_vector([per_country_features[c] for c in countries])
    for country in countries:
        region_signal_records.append({
            "app_id": app_id,
            "country": country,
            "country_deviation_from_app_centroid": normalized_l1_distance(per_country_features[country], centroid),
        })

region_signal_df = (
    pd.DataFrame(region_signal_records)
    .merge(country_meta, on=["app_id", "country"], how="left")
    .merge(app_group_lookup, on="app_id", how="inner")
    .dropna(subset=["region", "adaptation_group", "country_deviation_from_app_centroid"])
)

region_signal_summary = (
    region_signal_df.groupby(["adaptation_group", "region"], observed=False)
    .agg(
        mean_deviation=("country_deviation_from_app_centroid", "mean"),
        median_deviation=("country_deviation_from_app_centroid", "median"),
        app_country_observations=("country_deviation_from_app_centroid", "size"),
        apps=("app_id", "nunique"),
    )
    .reindex(
        pd.MultiIndex.from_product([adaptation_order, REGION_ORDER], names=["adaptation_group", "region"]),
        fill_value=0,
    )
    .reset_index()
)

region_signal_pivot = (
    region_signal_summary.pivot(index="adaptation_group", columns="region", values="mean_deviation")
    .reindex(index=adaptation_order, columns=REGION_ORDER, fill_value=0)
)

# Diagnostic table: raw observation composition is expected to be similar when
# each app is measured in the same countries. Keep it for verification only; it
# is not used as evidence of regional adaptation.
raw_region_count_pivot = (
    country_meta.merge(app_group_lookup, on="app_id", how="inner")
    .groupby(["adaptation_group", "region"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(index=adaptation_order, columns=REGION_ORDER, fill_value=0)
)
raw_region_pct_pivot = raw_region_count_pivot.div(
    raw_region_count_pivot.sum(axis=1).replace(0, np.nan), axis=0
).fillna(0) * 100

print("Raw app-country region composition, for diagnostic purposes only:")
display(raw_region_pct_pivot.round(2))
print("Region-level adaptation signal used in Figure (c):")
display(region_signal_summary.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2.5))
bars = ax.bar(
    group_counts["adaptation_group"],
    group_counts["apps"],
    color=[group_colors[g] for g in group_counts["adaptation_group"]]
)

for bar, pct in zip(bars, group_counts["percent"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

ax.set_title("Apps by Adaptation Group (Primary: K-means)")
ax.set_xlabel("")
ax.set_ylabel("Number of apps")
plt.tight_layout()
plt.savefig(FIG_DIR / "adaptation_group_counts_kmeans.png", bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.grid": True,
    "grid.color": "#CCCCCC",
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "legend.frameon": False,
})

PANEL_TITLE_SIZE = 14
LABEL_SIZE = 14
TICK_SIZE = 14
POINT_ALPHA = 0.65
GRID_ALPHA = 0.7
LINE_WIDTH = 2.2
MARKER_SIZE = 36

def style_ax(ax, title=None, xlabel=None, ylabel=None, grid=True):
    if title:
        ax.set_title(title, fontsize=PANEL_TITLE_SIZE, pad=8)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=LABEL_SIZE)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=LABEL_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_SIZE)
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(grid, linestyle="--", linewidth=0.6, alpha=GRID_ALPHA)

# -------------------------------------------------------------------
# COMPOSITE FIGURE 1:
# (a) AS histogram with K-means centroids
# (b) AS vs countries observed
# (c) Region-level adaptation signal from within-app country deviations
# -------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor("white")

# -----------------------------
# (a) Histogram of AS
# -----------------------------
ax = axes[0]
ax.hist(adaptation_app_level["AS"], bins=30, edgecolor="white", alpha=0.9, color="#B0BEC5")

centroid_styles = {
    "Stable": {"color": group_colors["Stable"], "linestyle": "--"},
    "Partially adaptive": {"color": group_colors["Partially adaptive"], "linestyle": "--"},
    "Highly adaptive": {"color": group_colors["Highly adaptive"], "linestyle": "--"},
}
for _, row in centroid_df.iterrows():
    label = cluster_to_label[row["cluster"]]
    style = centroid_styles[label]
    ax.axvline(
        row["centroid_AS"], color=style["color"], linestyle=style["linestyle"],
        linewidth=LINE_WIDTH, label=f"{label} centroid = {row['centroid_AS']:.3f}"
    )

style_ax(ax, title="(a) Distribution of App-Level AS", xlabel="Adaptation Score (AS)", ylabel="Number of apps")
ax.legend(fontsize=14, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=1)

# -----------------------------
# (b) AS vs countries observed
# -----------------------------
ax = axes[1]
for grp in adaptation_order:
    subset = adaptation_app_level.loc[adaptation_app_level["adaptation_group"] == grp]
    ax.scatter(subset["n_countries"], subset["AS"], s=MARKER_SIZE, alpha=POINT_ALPHA, label=grp, color=group_colors[grp])

style_ax(ax, title="(b) Behavioral Adaptation vs # Countries", xlabel="Countries observed", ylabel="Adaptation Score (AS)")
ax.legend(fontsize=14, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=1)

# -----------------------------
# (c) Region-level adaptation signal
# -----------------------------
ax = axes[2]

# Lighter + brighter custom palette (soft, high-contrast)
region_color_map_light = {
    "North America": "#6BAED6",   # light blue
    "Latin America": "#FDAE6B",   # light orange
    "Europe": "#FB6A4A",          # light red
    "Middle East": "#C49C94",     # soft brown
    "Asia": "#BC80BD",            # soft purple
    "Africa": "#BFD84D",          # light olive/green
    "Oceania": "#66C2A5",         # light teal
}

x = np.arange(len(adaptation_order))
bar_width = 0.11

regions_to_plot = [r for r in REGION_ORDER if r in region_signal_pivot.columns]
start = -bar_width * (len(regions_to_plot) - 1) / 2

for i, region in enumerate(regions_to_plot):
    vals = region_signal_pivot.loc[adaptation_order, region].values
    ax.bar(
        x + start + i * bar_width,
        vals,
        width=bar_width,
        label=region,
        color=region_color_map_light.get(region, "#CCCCCC"),
        edgecolor="white",   # improves readability
        linewidth=0.8,
        alpha=0.95           # slightly softer
    )

adaptation_order_multiline = ["Stable", "Partially\nAdaptive", "Highly\nAdaptive"]
ax.set_xticks(x)
ax.set_xticklabels(adaptation_order_multiline)

style_ax(
    ax,
    title="(c) Regional Adaptation Signal",
    ylabel="Mean within-app regional deviation",
)

ax.legend(
    fontsize=13,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=3,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / "adaptation_overview_subfigures.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


In [ ]:
# -------------------------------------------------------------------
# COMPOSITE FIGURE 2:
# Adaptation relationships
# -------------------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patheffects as pe

# -----------------------------
# GLOBAL STYLE FUNCTION (UPDATED)
# -----------------------------
def style_ax(ax, title="", xlabel="", ylabel=""):
    ax.set_title(title, fontsize=20, pad=12)          # ↑ bigger subtitle
    ax.set_xlabel(xlabel, fontsize=18, labelpad=8)    # ↑ bigger x-label
    ax.set_ylabel(ylabel, fontsize=18, labelpad=8)    # ↑ bigger y-label

    ax.tick_params(axis="both", which="major", labelsize=14)

    ax.grid(True, linestyle="--", alpha=0.7)

    # publication style cleanup
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# -----------------------------
# CREATE FIGURE
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.patch.set_facecolor("white")
axes = axes.flatten()

# -----------------------------
# (a) Privacy metric profiles
# -----------------------------
ax = axes[0]

group_metric_summary = (
    adaptation_app_level.groupby("adaptation_group", observed=False)
    .agg(
        ADII_mean=("ADII_mean", "mean"),
        DGI_mean=("DGI_mean", "mean"),
        PCLR_mean=("PCLR_mean", "mean"),
        AS_mean=("AS", "mean"),
        apps=("app_id", "nunique"),
    )
    .reindex(adaptation_order)
    .reset_index()
)

metrics_for_profile = ["ADII_mean", "DGI_mean", "PCLR_mean", "AS_mean"]

profile = group_metric_summary[metrics_for_profile].copy()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min()).replace(0, np.nan)
profile_norm = profile_norm.fillna(0)

x = np.arange(len(metrics_for_profile))

for idx, grp in enumerate(adaptation_order):
    ax.plot(
        x,
        profile_norm.iloc[idx].tolist(),
        marker="o",
        linewidth=LINE_WIDTH,
        markersize=8,
        label=grp,
        color=group_colors[grp],
    )

style_ax(
    ax,
    title="(a) Privacy Metric Profiles by\nAdaptation Group",
    xlabel="Privacy Metrics (Mean Values)",
    ylabel="Normalized mean value",
)

ax.set_xticks(x)
ax.set_xticklabels(["ADII", "DGI", "PCLR", "AS"], fontsize=15)

ax.legend(
    fontsize=17,
    loc="upper center",
    bbox_to_anchor=(0.4, -0.22),
    ncol=1,
    labelspacing=0,
    handletextpad=0.3,
    columnspacing=0.3,
    borderpad=0.2,
)

# -----------------------------
# (b) AS vs DGI
# -----------------------------
ax = axes[1]

plot_df = adaptation_app_level.copy()

for grp in adaptation_order:
    sub = plot_df.loc[plot_df["adaptation_group"] == grp]
    ax.scatter(
        sub["AS"],
        sub["DGI_mean"],
        s=MARKER_SIZE,
        alpha=POINT_ALPHA,
        label=grp,
        color=group_colors[grp],
    )

valid = plot_df[["AS", "DGI_mean"]].dropna()

if len(valid) >= 2:
    z = np.polyfit(valid["AS"], valid["DGI_mean"], 1)
    p = np.poly1d(z)
    xline = np.linspace(valid["AS"].min(), valid["AS"].max(), 200)
    ax.plot(xline, p(xline), linestyle="--", linewidth=LINE_WIDTH, color="black")

corr_as_dgi = valid["AS"].corr(valid["DGI_mean"]) if len(valid) >= 2 else np.nan

style_ax(
    ax,
    title=f"(b) Adaptation vs Disclosure Gap\n(r = {corr_as_dgi:.2f})",
    xlabel="Adaptation Score (AS)",
    ylabel="Mean DGI",
)

ax.legend(
    fontsize=17,
    loc="upper center",
    bbox_to_anchor=(0.4, -0.22),
    ncol=1,
    labelspacing=0,
    handletextpad=0.3,
    columnspacing=0.3,
    borderpad=0.2,
)

# -----------------------------
# (c) AS vs PCLR by region
# -----------------------------
ax = axes[2]

region_color_map_light = {
    "North America": "#4CC9F0",
    "Latin America": "#F9A826",
    "Europe": "#FF6B6B",
    "Middle East": "#B08968",
    "Asia": "#D77BE5",
    "Africa": "#A7C957",
    "Oceania": "#48CAE4",
}

app_as_lookup = adaptation_app_level[["app_id", "AS"]].drop_duplicates()

valid_region = (
    analysis_df[["app_id", "country", "country_label", "region", "PCLR"]]
    .merge(app_as_lookup, on="app_id", how="inner")
    .dropna(subset=["AS", "PCLR", "region"])
    .drop_duplicates(subset=["app_id", "country", "region"])
)

as_med = valid_region["AS"].median()
pclr_med = valid_region["PCLR"].median()

for region in REGION_ORDER:
    sub = valid_region.loc[valid_region["region"] == region]
    if sub.empty:
        continue

    ax.scatter(
        sub["AS"],
        sub["PCLR"],
        s=MARKER_SIZE,
        alpha=0.65,
        label=region,
        color=region_color_map_light.get(region, "#BBBBBB"),
        edgecolor="white",
        linewidth=0.4,
    )

if len(valid_region) >= 2:
    z = np.polyfit(valid_region["AS"], valid_region["PCLR"], 1)
    p = np.poly1d(z)
    xline = np.linspace(valid_region["AS"].min(), valid_region["AS"].max(), 200)
    ax.plot(xline, p(xline), linestyle="--", linewidth=LINE_WIDTH, color="black")

ax.axvline(as_med, linestyle="--", linewidth=1.8, color="black", alpha=0.7)
ax.axhline(pclr_med, linestyle="--", linewidth=1.8, color="black", alpha=0.7)

# quadrant labels
x_min, x_max = valid_region["AS"].min(), valid_region["AS"].max()
y_min, y_max = valid_region["PCLR"].min(), valid_region["PCLR"].max()

def add_label(x_pos, y_pos, text):
    t = ax.text(
        x_pos,
        y_pos,
        text,
        fontsize=12,
        weight="bold",
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#F0F0F0", edgecolor="none", alpha=0.85),
        zorder=5,
    )
    t.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])

add_label((x_min + as_med)/2, (pclr_med + y_max)/2, "Stable & Leaky")
add_label((as_med + x_max)/2, (pclr_med + y_max)/2, "Adaptive & Leaky")
add_label((x_min + as_med)/2, (y_min + pclr_med)/2, "Stable & Safer")
add_label((as_med + x_max)/2, (y_min + pclr_med)/2, "Adaptive & Safer")

# App-level correlation (one row per app: AS vs PCLR_mean), matching panel
# (b)'s level of analysis -- see note above the scatter data for why this
# must not be computed from the pseudo-replicated valid_region table.
app_level_corr_df = adaptation_app_level[["AS", "PCLR_mean"]].dropna()
corr_as_pclr_region = (
    app_level_corr_df["AS"].corr(app_level_corr_df["PCLR_mean"])
    if len(app_level_corr_df) >= 2 else np.nan
)

style_ax(
    ax,
    title=f"(c) Adaptation vs Pre-Consent Leakage\n(r = {corr_as_pclr_region:.2f}, app-level)",
    xlabel="Adaptation Score (AS)",
    ylabel="Country-level PCLR",
)

ax.legend(
    fontsize=17,
    loc="upper center",
    bbox_to_anchor=(0.4, -0.22),
    ncol=3,
    labelspacing=0,
    handletextpad=0.3,
    columnspacing=0.3,
    borderpad=0.2,
)

# -----------------------------
# (d) Downloads vs AS
# -----------------------------
ax = axes[3]

plot_df_downloads = adaptation_app_level.dropna(
    subset=["downloads_int", "AS", "adaptation_group"]
)
plot_df_downloads = plot_df_downloads[plot_df_downloads["downloads_int"] > 0]

for grp in adaptation_order:
    sub = plot_df_downloads[plot_df_downloads["adaptation_group"] == grp]
    ax.scatter(
        sub["downloads_int"],
        sub["AS"],
        s=MARKER_SIZE,
        alpha=POINT_ALPHA,
        label=grp,
        color=group_colors[grp],
    )

ax.set_xscale("log")

style_ax(
    ax,
    title="(d) Popularity vs Geographic Adaptation\n",
    xlabel="Downloads (log scale)",
    ylabel="Adaptation Score (AS)",
)

ax.legend(
    fontsize=17,
    loc="upper center",
    bbox_to_anchor=(0.4, -0.22),
    ncol=1,
    labelspacing=0,
    handletextpad=0.3,
    columnspacing=0.3,
    borderpad=0.2,
)

# -----------------------------
# FINALIZE
# -----------------------------
plt.tight_layout()
plt.savefig(FIG_DIR / "adaptation_relationships_subfigures.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()